### Generic ETL Notebook – Dynamic File Processing

**Goal:** Build a reusable ETL process capable of processing any supported source file (`torneio.csv`, `estadios.csv`, or others), making the data available in the appropriate layer.

The source file is defined dynamically via widget parameter, allowing this notebook to be reused without modification.

---

### How It Works

This notebook accepts any supported CSV file as input through the `my_source_file` widget.  
Depending on the value provided, the ETL will process the corresponding dataset.

**Supported files (examples):**
- `torneio.csv`
- `estadios.csv`

---

### ETL Pipeline

Read → Transform → Write

1. Read the CSV file from the data lake using the dynamic path defined by `my_source_file`
2. Define the correct data schema for the file being processed
3. Include a column with the date when the file was ingested (`ingestion_date`)
4. Save the data in **Parquet** format in the appropriate layer

---

### Widget Configuration
```python
dbutils.widgets.text("source_file", "")
my_source_file = dbutils.widgets.get("source_file")

my_source_file
```

> **How to use:** When running this notebook, set the `source_file` parameter to the desired file name (e.g., `torneio.csv` or `estadios.csv`).  
> The notebook will automatically adapt the ETL process to the provided file.

In [0]:
dbutils.widgets.text("source_file","")
my_source_file = dbutils.widgets.get("source_file")


In [0]:
my_source_file

In [0]:
%run "../Modulo 3//00_config_storage"

In [0]:
%run "../Modulo 4/Functions"

In [0]:
%run "../Modulo 4/Variables"

In [0]:
display(dbutils.fs.ls(path_bronze))

In [0]:
path_source_file = f"{path_bronze}/{my_source_file}.csv"

In [0]:
df = spark.read.format("csv") \
.option("header", "true") \
.option("encoding","UTF-16") \
.option("inferSchema", "true") \
.load(path_source_file)

display(df)
df.printSchema()


In [0]:
from pyspark.sql.types import BooleanType
if my_source_file == "torneios":
    df_2 = df.withColumn(
        "season_current", df.season_current.cast(BooleanType())
    )
    df_2.printSchema()
else:
    df_2 = df.withColumnRenamed("id", "stadium_id") \
                               .withColumnRenamed("name", "stadium_name") \
                               .withColumnRenamed("address", "stadium_address") \
                               .withColumnRenamed("city", "stadium_city") \
                               .withColumnRenamed("country", "stadium_country") \
                               .withColumnRenamed("capacity", "stadium_capacity") \
                               .withColumnRenamed("surface", "stadium_surface") \
                               .withColumnRenamed("image", "stadium_image")

display(df_2)
df_2.printSchema()

In [0]:
df_date = create_column_date(df_2)
display(df_date)

In [0]:
df_date.write.mode("overwrite").parquet(f"{path_silver}/{my_source_file}")